# KNN

Clean, final version of the collaborative-filtering component. Produces a directional
rater→partner score `score_cf` for **every** pair, to feed the team's contextual FM.

- **Protocol:** "sidestep wave isolation" — 3-fold cross-val over predefined groups of
  whole waves (train on 2 folds, test on the held-out fold, rotate). Each person is in
  exactly one wave, so train and test never share a person, and every pair is scored
  exactly once (100% coverage).
- **Model:** KNN, **k=60, distance-weighted, Manhattan metric** — the configuration that
  scored best when tuned on this split.
- **Target:** `dec`.  **Eval:** AUC-ROC + LogLoss.
- **Output:** `score_cf_sidestep.csv` (`iid, pid, dec, fold, score_cf`).

In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.base import clone

In [2]:
# ---- build the pair table (rater profile + partner profile) ---------------
df = pd.read_csv('dataset/speed_dating_clean.csv')
df['pid'] = df['pid'].astype(int)

roles = json.load(open('dataset/column_roles.json'))
feature_cols = roles['feature_cols']

partner = ['age_o','race_o','int_corr','samerace','met_before','met_before_o','age_diff','race_match','scale_type']
user = [c for c in feature_cols if c not in partner]

profile = df[['iid'] + user].drop_duplicates('iid')
pairs = df.merge(profile, left_on='pid', right_on='iid', suffixes=('', '_cand'))
pairs.shape

(7976, 166)

In [3]:
# ---- feature columns (numeric attributes only) ----------------------------
cat = ['field_cd', 'career_c', 'race']
num_user = [c for c in user if c not in cat]
num_cand = [c + '_cand' for c in num_user]
pair_feat = ['int_corr','samerace','age_diff','race_match','met_before','met_before_o']
num_cols = num_user + num_cand + pair_feat

X = pairs[num_cols]
y = pairs['dec']

# identity / grouping keys
waves   = pairs['wave']
rater   = pairs['iid'].to_numpy()   # who is judging
partner = pairs['pid'].to_numpy()   # who is being judged

In [4]:
# ---- final model: KNN k=60, distance-weighted, Manhattan ------------------
model = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), num_cols)])),
    ('knn', KNeighborsClassifier(n_neighbors=60, weights='distance', metric='manhattan'))])

In [5]:
# ---- sidestep wave isolation: 3-fold CV over predefined wave groups -------
wave_folds = {1: [1, 4, 5, 17, 19, 20, 21],
              2: [3, 6, 7, 11, 13, 15, 18],
              3: [2, 8, 9, 10, 14, 16]}
fold_of = {w: f for f, ws in wave_folds.items() for w in ws}
fold = waves.map(fold_of).to_numpy()

recs = []
for f in sorted(set(fold)):
    te = np.where(fold == f)[0]; tr = np.where(fold != f)[0]
    mdl = clone(model).fit(X.iloc[tr], y.iloc[tr])
    recs.append(pd.DataFrame({'iid': rater[te], 'pid': partner[te],
                              'dec': y.iloc[te].to_numpy(), 'fold': f,
                              'score_cf': mdl.predict_proba(X.iloc[te])[:, 1]}))
score_cf = pd.concat(recs, ignore_index=True)

In [ ]:
# ---- evaluate (target = dec) and save -------------------------------------
auc = roc_auc_score(score_cf['dec'], score_cf['score_cf'])
ll  = log_loss(score_cf['dec'], score_cf['score_cf'])
print(f"{len(score_cf)} rows = every pair, 100% coverage")
print(f"target=dec   AUC-ROC={auc:.3f}   LogLoss={ll:.3f}")
for f in sorted(set(fold)):
    s = score_cf[score_cf['fold'] == f]
    print(f"   fold {f} (waves {wave_folds[f]}): n={len(s)}  AUC={roc_auc_score(s['dec'], s['score_cf']):.3f}")

score_cf.to_csv('score_cf_sidestep.csv', index=False)
print('saved -> score_cf_sidestep.csv')
score_cf.head()

7976 rows = every pair, 100% coverage
target=dec   AUC-ROC=0.601   LogLoss=0.667
   fold 1 (waves [1, 4, 5, 17, 19, 20, 21]): n=2810  AUC=0.594
   fold 2 (waves [3, 6, 7, 11, 13, 15, 18]): n=2580  AUC=0.603
   fold 3 (waves [2, 8, 9, 10, 14, 16]): n=2586  AUC=0.612
saved -> score_cf_sidestep.csv


,iid,pid,dec,fold,score_cf
0,1,11,1,1,0.283502
1,1,12,1,1,0.419684
2,1,13,1,1,0.371016
3,1,14,1,1,0.349775
4,1,15,1,1,0.319025
